# MP POTENTIAL CHAIN (Clive logic)

In [12]:
import pandas as pd
import numpy as np
df_hotel_list = pd.read_csv("C:\\Users\\svi02\\Documents\\misc\\CSV\\hotelLIST1.csv" ,
                 sep=",",
                 encoding = 'unicode_escape'
                   )
 #df_hotel_list = {'HOTEL_ID':['171243']}

df_hotel_list = pd.DataFrame(df_hotel_list)
#df_hotel_list.head()
df_corsa = pd.DataFrame()
a=[]

In [13]:
df_hotel_list.count()

HOTEL_ID    130
dtype: int64

In [14]:
import pyexasol
import configparser

#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\svi02\\.spyder-py3\\ExasolDET.ini')

dsn=config['exasolDET']['dsn']
user=config['exasolDET']['user']
pwd=config['exasolDET']['pwd']
schema=config['exasolDET']['schema']
# Exasol connection
connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)

#171243

ExaConnectionFailedError: 
(
    message  =>  Could not connect to Exasol: timed out
    dsn      =>  10.150.112.211..215:8563
    user     =>  app_bus07
    schema   =>  
)


In [11]:
# PASTE EXASOL QUERY HERE


for  i in range(0, len(df_hotel_list.HOTEL_ID)) :
    sql_query =  """

     select az.hotel_id, az.hotel_name, az.prio, az.HOTEL_CHAIN_ID, az.HOTEL_CHAIN_NAME, az.CATEGORY, az.AVG_RATING , az.CAPACITY, az.ANALYZED_HOTEL_RNP_BY_SOURCING_CLIENTS_2018,
    
    az.OCCUPANCY_RATE_BY_SOURCING_CLIENTS_2018*100,
    az.CONTRACT_STATUS, az.HOTEL_STATUS, az.CITY_ID, az.CITY_NAME, az.RFP_DESTINATION, az.COUNTRY_ID, az.COUNTRY_NAME, az.COUNTRY_CODE, az.WORLD_REGION, az.ACCEPTED_SOURCING_CLIENTS_2018, az.HOTEL_CURRENCY, 
    az.N_AVG_ACCEPTED_RATE_2018, az.N_MIN_ACCEPTED_RATE_2018, az.N_AVG_SUBMITTED_RATE_2018, az.N_MIN_SUBMITTED_RATE_2018, az.N_BOTTOM_UP_MIN_TARGET_RATE, az.N_BOTTOM_UP_MAX_TARGET_RATE, az.N_BOTTOM_UP_AVG_TARGET_RATE,
    az.SMART_BID, az.CCR, az.SYSTEM_BENCHMARK_RATE, az.DISTANCE_TO_VIRTUAL_POI, az.VIRTUAL_POI_NAME, az.POI_AVG_CAPACITY , 
   az.CPOICOUNT, az.TARGET_RATE_CURRENCY, az.N_TARGET_RATE_LRA_INCL_BF,
    
    sum(az.count_PROB_AMEND) as VPOI_SPECIFIC ,
    sum(az.PROB_AMEND) as PROB_AMEND_D,
    local.PROB_AMEND_D / nullif(local.VPOI_SPECIFIC , 0) as  Avg_RNs_per_Customer,
    (local.PROB_AMEND_D/200/az.capacity) * 100 as Incremental_RNs_Occupancy_Rate

    
    
    from (

select a.hotel_id, a.hotel_name,a.HOTEL_CHAIN_ID, b.PRIO, ch.HOTEL_CHAIN_NAME, b.CATEGORY, b.AVG_RATING , b.CAPACITY, b.ANALYZED_HOTEL_RNP_BY_SOURCING_CLIENTS_2018,
    
    b.OCCUPANCY_RATE_BY_SOURCING_CLIENTS_2018,--cli.CLIENT_NAME,
    b.CONTRACT_STATUS, b.HOTEL_STATUS, b.CITY_ID, b.CITY_NAME, b.RFP_DESTINATION, b.COUNTRY_ID, b.COUNTRY_NAME, b.COUNTRY_CODE, b.WORLD_REGION, b.ACCEPTED_SOURCING_CLIENTS_2018, b.HOTEL_CURRENCY, 
     b.AVG_ACCEPTED_RATE_2018 as N_AVG_ACCEPTED_RATE_2018, b.MIN_ACCEPTED_RATE_2018 as N_MIN_ACCEPTED_RATE_2018, 
    
     b.AVG_SUBMITTED_RATE_2018 as N_AVG_SUBMITTED_RATE_2018,  b.MIN_SUBMITTED_RATE_2018
     as N_MIN_SUBMITTED_RATE_2018
     ,cur.CURRENCY_EXCHANGE_RATE *  b.BOTTOM_UP_MIN_TARGET_RATE as N_BOTTOM_UP_MIN_TARGET_RATE,cur.CURRENCY_EXCHANGE_RATE *  b.BOTTOM_UP_MAX_TARGET_RATE as N_BOTTOM_UP_MAX_TARGET_RATE,
     cur.CURRENCY_EXCHANGE_RATE *  b.BOTTOM_UP_AVG_TARGET_RATE as N_BOTTOM_UP_AVG_TARGET_RATE,
    b.SMART_BID, b.CCR, b.SYSTEM_BENCHMARK_RATE, b.DISTANCE_TO_VIRTUAL_POI, c.HOTELS_ALL as POI_AVG_CAPACITY ,
    
    
   cou.CPOICOUNT,
    
    
    b.TARGET_RATE_CURRENCY,    b.TARGET_RATE_LRA_INCL_BF as N_TARGET_RATE_LRA_INCL_BF,
    
   inn.VIRTUAL_POI_NAME , inn.CLIENT_NAME , inn.VPOI_RN_SPELLED_OUT_2018, inn.FORECAST_REQUIRED_HOTELS, inn.SPELLED_OUT_RNS_2018_FORECAST_REQUIRED_HOTELS,
inn.OTHER_PRIO_INCREMENTAL_HOTELS_ON_LEAD_LIST_PER_CLIENTVPOI,
inn.CLIENT_ACCEPTED_AVG_CATEGORY, inn.CLIENT_ACCEPTED_AVG_RATING_HRS_GOOGLE, cur.CURRENCY_EXCHANGE_RATE * coalesce(inn.CLIENT_ACCEPTED_AVG_RATE_2018, 0) as N_CLIENT_ACCEPTED_AVG_RATE_2018,

     cur.CURRENCY_EXCHANGE_RATE * coalesce(inn.CLIENT_ACCEPTED_MIN_RATE_2018 , 0) as N_CLIENT_ACCEPTED_MIN_RATE_2018,
case when inn.hotel_id || ' - ' || inn.client_name = cli.ID then 'Yes' else 'No' end as hotel_invited ,

hot.SUBMITTED_RATE_TYPE , coalesce(hot.SUBMITTED_RATE, 0) as SUBMITTED_RATE , hot.RFP_HOTEL_STATUS_2018,



case when hot.RFP_HOTEL_STATUS_2018 is not null then 
      case when local.N_CLIENT_ACCEPTED_MIN_RATE_2018 = 0 and local.N_CLIENT_ACCEPTED_AVG_RATE_2018 = 0 then 0.5 
     else case when local.N_TARGET_RATE_LRA_INCL_BF < local.N_CLIENT_ACCEPTED_MIN_RATE_2018 then 0.9 
  else      case when local.N_TARGET_RATE_LRA_INCL_BF < local.N_CLIENT_ACCEPTED_AVG_RATE_2018 then 0.8
  else   case when coalesce(local.N_CLIENT_ACCEPTED_AVG_RATE_2018/local.N_TARGET_RATE_LRA_INCL_BF, 0) < 0.5  then 0
  else 0.5 *  (local.N_CLIENT_ACCEPTED_AVG_RATE_2018/local.N_TARGET_RATE_LRA_INCL_BF) 
  
 end end  end   end end   as    numb,
    
 case when hot.RFP_HOTEL_STATUS_2018 is  null then 


      case when  local.N_CLIENT_ACCEPTED_MIN_RATE_2018 = 0 and local.N_CLIENT_ACCEPTED_AVG_RATE_2018 = 0 then 0.4
         else case when    local.N_TARGET_RATE_LRA_INCL_BF < local.N_CLIENT_ACCEPTED_MIN_RATE_2018 then 0.8 
            else case when    local.N_TARGET_RATE_LRA_INCL_BF < local.N_CLIENT_ACCEPTED_AVG_RATE_2018 then 0.7
                 else  case when   coalesce(local.N_CLIENT_ACCEPTED_AVG_RATE_2018/local.N_TARGET_RATE_LRA_INCL_BF, 0) < 0.5  then 0
                else  0.25 *  (local.N_CLIENT_ACCEPTED_AVG_RATE_2018/local.N_TARGET_RATE_LRA_INCL_BF)
end end  end   end end   as    numb2,

 case when hot.RFP_HOTEL_STATUS_2018 is not null then  local.numb else local.numb2 end as INVITED_PROB,

 local.INVITED_PROB * 100 as INVITATION_ON_PROB, 
 
 SPELLED_OUT_RNS_2018_FORECAST_REQUIRED_HOTELS * local.INVITED_PROB as PROB_AMEND,

 case when local.INVITATION_ON_PROB > 50  then 1 else 0 end as count_PROB_AMEND



    from DWHBIL.V_LKP_HOTEL a 
left join TEMP.CLI_MP_LEAD_LIST b on a.hotel_id = b.HOTEL_ID
left join TEMP.CLI_POI_TABLE c on b.VIRTUAL_POI_NAME = c.VIRTUAL_POI_NAME
left join (

select count(*) as CPOICOUNT, VIRTUAL_POI_NAME from TEMP.CLI_MP_LEAD_LIST k where PRIO <=3 group by VIRTUAL_POI_NAME

) cou on cou.VIRTUAL_POI_NAME = b.VIRTUAL_POI_NAME



join (


 
select a.hotel_id ,c.VIRTUAL_POI_NAME , cpo.CLIENT_NAME , cpo.VPOI_RN_SPELLED_OUT_2018, cpo.FORECAST_REQUIRED_HOTELS, cpo.SPELLED_OUT_RNS_2018_FORECAST_REQUIRED_HOTELS,
cpo.OTHER_PRIO_INCREMENTAL_HOTELS_ON_LEAD_LIST_PER_CLIENTVPOI,
cpo.CLIENT_ACCEPTED_AVG_CATEGORY, cpo.CLIENT_ACCEPTED_AVG_RATING_HRS_GOOGLE, cpo.CLIENT_NAME || ' - ' || c.VIRTUAL_POI_NAME as CLI_VPOI, hc.CLIENT_ACCEPTED_AVG_RATE_2018, hc.CLIENT_ACCEPTED_MIN_RATE_2018




  from   DWHBIL.V_LKP_HOTEL a 
 join TEMP.CLI_MP_LEAD_LIST b on a.hotel_id = b.HOTEL_ID
join TEMP.CLI_POI_TABLE c on b.VIRTUAL_POI_NAME = c.VIRTUAL_POI_NAME
join TEMP.CLI_ACCEPTED_CLIENT ac on 1 =1  --and ac.CLIENT_NAME ='CRRC' OTHER_PRIO_INCREMENTAL_HOTELS_ON_LEAD_LIST_PER_CLIENTVPOI

   join TEMP.CLI_CLIENTS_PER_VPOI_LATEST_DACH cpo on cpo.VPOI_NAME = c.VIRTUAL_POI_NAME  and replace(ac.CLIENT_NAME, CHAR(13), '') <> replace(cpo.CLIENT_NAME, CHAR(13), '')
   join  TEMP.CLI_HOTEL_CLIENT_DACH hc on hc.CLIENT_VPOI = cpo.CLIENT_NAME || ' - ' || c.VIRTUAL_POI_NAME --and hc.HOTEL_ID = a.HOTEL_ID
  
   
where cpo.CLIENT_NAME not in (
select 
cpo.CLIENT_NAME
    from DWHBIL.V_LKP_HOTEL a 
 join TEMP.CLI_MP_LEAD_LIST b on a.hotel_id = b.HOTEL_ID
 join TEMP.CLI_POI_TABLE c on b.VIRTUAL_POI_NAME = c.VIRTUAL_POI_NAME
 join TEMP.CLI_ACCEPTED_CLIENT ac on   1 = 1
   join TEMP.CLI_CLIENTS_PER_VPOI_LATEST_DACH cpo on cpo.VPOI_NAME = c.VIRTUAL_POI_NAME  and ac.HOTEL_ID || '-' || replace(ac.CLIENT_NAME, CHAR(13), '') =  a.HOTEL_ID || '-' || replace(cpo.CLIENT_NAME, CHAR(13), '')
 
 where 
 a.hotel_id ="""+ str(df_hotel_list.HOTEL_ID[i])+"""



  group by cpo.CLIENT_NAME) 
 and   a.hotel_id ="""+ str(df_hotel_list.HOTEL_ID[i])+"""


  group by a.hotel_id ,c.VIRTUAL_POI_NAME , cpo.CLIENT_NAME , cpo.VPOI_RN_SPELLED_OUT_2018, cpo.FORECAST_REQUIRED_HOTELS, cpo.SPELLED_OUT_RNS_2018_FORECAST_REQUIRED_HOTELS,
cpo.OTHER_PRIO_INCREMENTAL_HOTELS_ON_LEAD_LIST_PER_CLIENTVPOI,
cpo.CLIENT_ACCEPTED_AVG_CATEGORY, cpo.CLIENT_ACCEPTED_AVG_RATING_HRS_GOOGLE, cpo.CLIENT_NAME || '-' || c.VIRTUAL_POI_NAME,
hc.CLIENT_ACCEPTED_AVG_RATE_2018, hc.CLIENT_ACCEPTED_MIN_RATE_2018
--hc.SUBMITTED_RATE_TYPE --, hc.SUBMITTED_RATE, hc.RFP_HOTEL_STATUS_2018

 ) inn on inn.hotel_id = a.hotel_id 
 
 
 
 left  join (

 
 
select hc.SUBMITTED_RATE_TYPE , hc.SUBMITTED_RATE, hc.RFP_HOTEL_STATUS_2018 ,



a.hotel_id ,c.VIRTUAL_POI_NAME , cpo.CLIENT_NAME 
  from   DWHBIL.V_LKP_HOTEL a 
 join TEMP.CLI_MP_LEAD_LIST b on a.hotel_id = b.HOTEL_ID
join TEMP.CLI_POI_TABLE c on b.VIRTUAL_POI_NAME = c.VIRTUAL_POI_NAME
join TEMP.CLI_ACCEPTED_CLIENT ac on  1 =1 
   join TEMP.CLI_CLIENTS_PER_VPOI_LATEST_DACH cpo on cpo.VPOI_NAME = c.VIRTUAL_POI_NAME  and replace(ac.CLIENT_NAME, CHAR(13), '') <> replace(cpo.VPOI_NAME_CLIENT_NAME, CHAR(13), '')
   join  TEMP.CLI_HOTEL_CLIENT_DACH hc on hc.id = a.HOTEL_ID || ' - ' ||  cpo.CLIENT_NAME 
  
   
where cpo.CLIENT_NAME not in (
select 
cpo.CLIENT_NAME
     from DWHBIL.V_LKP_HOTEL a 
 join TEMP.CLI_MP_LEAD_LIST b on a.hotel_id = b.HOTEL_ID
 join TEMP.CLI_POI_TABLE c on b.VIRTUAL_POI_NAME = c.VIRTUAL_POI_NAME
 join TEMP.CLI_ACCEPTED_CLIENT ac on   1 = 1
   join TEMP.CLI_CLIENTS_PER_VPOI_LATEST_DACH cpo on cpo.VPOI_NAME = c.VIRTUAL_POI_NAME  and ac.HOTEL_ID || '-' || replace(ac.CLIENT_NAME, CHAR(13), '') =  a.HOTEL_ID || '-' || replace(cpo.CLIENT_NAME, CHAR(13), '')
 
 where 

    a.hotel_id ="""+ str(df_hotel_list.HOTEL_ID[i])+"""


  group by cpo.CLIENT_NAME) 
 and      a.hotel_id ="""+ str(df_hotel_list.HOTEL_ID[i])+"""

  group by a.hotel_id ,c.VIRTUAL_POI_NAME , cpo.CLIENT_NAME ,
hc.SUBMITTED_RATE_TYPE , hc.SUBMITTED_RATE, hc.RFP_HOTEL_STATUS_2018

 ) hot on hot.hotel_id = a.hotel_id and hot.CLIENT_NAME =  inn.CLIENT_NAME

join   TEMP.CLI_CURRENCY_CLI cur on cur.CURRENCY_ISO = TARGET_RATE_CURRENCY
left join TEMP.CLI_HOTEL_CLIENT_DACH cli on cli.HOTEL_ID = a.HOTEL_ID and inn.CLI_VPOI = cli.CLIENT_VPOI
  join DWHBIL.LKP_HOTEL_CHAIN ch on ch.HOTEL_CHAIN_ID = a.HOTEL_CHAIN_ID
where a.hotel_id ="""+ str(df_hotel_list.HOTEL_ID[i])+"""

group by 
a.hotel_id, a.hotel_name, a.hotel_chain_id, b.PRIO, ch.HOTEL_CHAIN_NAME, b.CATEGORY, b.AVG_RATING , b.CAPACITY, b.ANALYZED_HOTEL_RNP_BY_SOURCING_CLIENTS_2018,
    
    b.OCCUPANCY_RATE_BY_SOURCING_CLIENTS_2018,
    b.CONTRACT_STATUS, b.HOTEL_STATUS, b.CITY_ID, b.CITY_NAME, b.RFP_DESTINATION, b.COUNTRY_ID, b.COUNTRY_NAME, b.COUNTRY_CODE, b.WORLD_REGION, b.ACCEPTED_SOURCING_CLIENTS_2018, b.HOTEL_CURRENCY, 
    local.N_AVG_ACCEPTED_RATE_2018, local.N_MIN_ACCEPTED_RATE_2018, 
    
     local.N_AVG_SUBMITTED_RATE_2018,  local.N_MIN_SUBMITTED_RATE_2018
     ,local.N_BOTTOM_UP_MIN_TARGET_RATE,local.N_BOTTOM_UP_MAX_TARGET_RATE,
     local.N_BOTTOM_UP_AVG_TARGET_RATE,
    b.SMART_BID, b.CCR, b.SYSTEM_BENCHMARK_RATE, b.DISTANCE_TO_VIRTUAL_POI, c.HOTELS_ALL , cou.CPOICOUNT,
    b.TARGET_RATE_CURRENCY, local.N_TARGET_RATE_LRA_INCL_BF,
    
   inn.VIRTUAL_POI_NAME , inn.CLIENT_NAME , inn.VPOI_RN_SPELLED_OUT_2018, inn.FORECAST_REQUIRED_HOTELS, inn.SPELLED_OUT_RNS_2018_FORECAST_REQUIRED_HOTELS,
inn.OTHER_PRIO_INCREMENTAL_HOTELS_ON_LEAD_LIST_PER_CLIENTVPOI,
inn.CLIENT_ACCEPTED_AVG_CATEGORY, inn.CLIENT_ACCEPTED_AVG_RATING_HRS_GOOGLE, local.N_CLIENT_ACCEPTED_AVG_RATE_2018,

     local.N_CLIENT_ACCEPTED_MIN_RATE_2018,  cli.CLIENT_NAME,
local.hotel_invited ,
 inn.CLIENT_NAME, a.HOTEL_ID,
hot.SUBMITTED_RATE_TYPE , local.SUBMITTED_RATE , hot.RFP_HOTEL_STATUS_2018
   ) az
  
group by 
az.hotel_id, az.hotel_name, az.PRIO, az.HOTEL_CHAIN_ID, az.HOTEL_CHAIN_NAME, az.CATEGORY, az.AVG_RATING , az.CAPACITY, az.ANALYZED_HOTEL_RNP_BY_SOURCING_CLIENTS_2018,
    
    az.OCCUPANCY_RATE_BY_SOURCING_CLIENTS_2018,
    az.CONTRACT_STATUS, az.HOTEL_STATUS, az.CITY_ID, az.CITY_NAME, az.RFP_DESTINATION, az.COUNTRY_ID, az.COUNTRY_NAME, az.COUNTRY_CODE, az.WORLD_REGION, az.ACCEPTED_SOURCING_CLIENTS_2018, az.HOTEL_CURRENCY, 
    az.N_AVG_ACCEPTED_RATE_2018, az.N_MIN_ACCEPTED_RATE_2018, az.N_AVG_SUBMITTED_RATE_2018, az.N_MIN_SUBMITTED_RATE_2018, az.N_BOTTOM_UP_MIN_TARGET_RATE, az.N_BOTTOM_UP_MAX_TARGET_RATE, az.N_BOTTOM_UP_AVG_TARGET_RATE,
    az.SMART_BID, az.CCR, az.SYSTEM_BENCHMARK_RATE, az.DISTANCE_TO_VIRTUAL_POI, az.VIRTUAL_POI_NAME, az.POI_AVG_CAPACITY ,
    az.TARGET_RATE_CURRENCY, az.N_TARGET_RATE_LRA_INCL_BF,  az.CPOICOUNT
"""
    df_corsa = connect.export_to_pandas(sql_query)

    # Importing data into a DataFrame
  
 #     for row in QUERY:
#         a.append(row)
#     #print(len(a))
#     df_corsa = pd.DataFrame(a)
    
     df_corsa.columns =  QUERY.col_names
    print(i)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x8a in position 0: invalid start byte

In [10]:
df_corsa = pd.DataFrame(a)
df_col_names = QUERY.col_names
df_corsa.columns = df_col_name

NameError: name 'df_col_name' is not defined

In [ ]:
import datetime
 
d = datetime.datetime.today()
df_corsa.to_excel('C:\\Users\\svi02\\Documents\\misc\\CSV\\RESULT2_'+str(d.strftime('%d-%m-%Y_%H%M'))+'.xlsx', 
          sheet_name='Results', 
          header = True,
          encoding='utf-8',
          index=False)